In [ ]:
import json, math, statistics
from pathlib import Path
import plotly.io as pio
import os
pio.renderers.default = "browser"

# --------- 1) Robust JSON -> list[dict] normalizer (edit here if needed) ---------
def _is_record_list(x):
    return isinstance(x, list) and (len(x) == 0 or isinstance(x[0], dict))

def extract_records(obj):
    """Return a list of dict records from a variety of common result JSON schemas.
    
    Supported patterns (common in benchmark dumps):
      - list[dict]
      - {'results': list[dict]} or {'episodes': list[dict]} or {'data': list[dict]}
      - {'runs': {'methodA': list[dict], ...}}  -> flattened with method key
      - dict-of-dicts keyed by episode_id -> converted to records
    """
    # Case 1: already a list of records
    if _is_record_list(obj):
        return obj

    # Case 2: wrapper keys
    if isinstance(obj, dict):
        for k in ("results", "episodes", "data", "records"):
            if k in obj and _is_record_list(obj[k]):
                return obj[k]

        # Case 3: dict-of-dicts keyed by episode_id
        # e.g., {"ep_0001": {..}, "ep_0002": {..}}
        if all(isinstance(v, dict) for v in obj.values()):
            recs = []
            for key, val in obj.items():
                r = dict(val)
                # keep the key as an identifier if not present
                r.setdefault("episode_id", key)
                recs.append(r)
            return recs

        # Case 4: nested 'runs' or 'methods'
        for k in ("runs", "methods", "agents"):
            if k in obj and isinstance(obj[k], dict):
                recs = []
                for method_name, payload in obj[k].items():
                    subrecs = extract_records(payload)
                    for r in subrecs:
                        rr = dict(r)
                        rr.setdefault("method", method_name)
                        recs.append(rr)
                if recs:
                    return recs

    raise ValueError("Unrecognized JSON schema. Please edit extract_records() to match your file format.")


def load_records(path: str, method_name: str):
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    recs = extract_records(obj)
    # attach method name if missing
    out = []
    for r in recs:
        rr = dict(r)
        rr.setdefault("method", method_name)
        out.append(rr)
    return out


def try_float(x):
    """Convert x to float if possible, else return None."""
    if x is None:
        return None
    if isinstance(x, (int, float)):
        if isinstance(x, bool):
            return None
        return float(x)
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return None
        try:
            return float(s)
        except ValueError:
            return None
    return None


def numeric_keys(records):
    """Infer candidate numeric metric keys from records."""
    keys = set()
    for r in records:
        for k, v in r.items():
            if k in ("method",):
                continue
            fv = try_float(v)
            if fv is not None and not math.isnan(fv) and not math.isinf(fv):
                keys.add(k)
    return sorted(keys)


def summarize_by_method(records, metric_keys):
    """Compute mean, std, count for each metric per method."""
    by_m = {}
    for r in records:
        m = r.get("method", "unknown")
        by_m.setdefault(m, []).append(r)

    summary = {}  # method -> metric -> stats
    for m, rs in by_m.items():
        summary[m] = {}
        for k in metric_keys:
            vals = []
            for r in rs:
                fv = try_float(r.get(k))
                if fv is None or math.isnan(fv) or math.isinf(fv):
                    continue
                vals.append(fv)
            if len(vals) == 0:
                continue
            mu = statistics.fmean(vals)
            sd = statistics.pstdev(vals) if len(vals) > 1 else 0.0
            summary[m][k] = {"mean": mu, "std": sd, "n": len(vals)}
    return summary

def load_split(episode_folder, split_name):
    path = os.path.join(episode_folder, f"{split_name}_set.txt")
    with open(path, "r") as f:
        return [line.strip() for line in f.readlines()]

def load_all_gt_data(episode_folder):
    all_episodes = {}
    for scene_file in os.listdir(episode_folder):
        if scene_file.endswith(".json"):
            scene_id = scene_file.split(".")[0]
            data = json.load(open(os.path.join(episode_folder, scene_file), "r"))
            data = [data[i] for i in range(len(data))]
            all_episodes[scene_id] = data
    return all_episodes


In [ ]:
# --------- 2) Load & merge all results in ONE block ---------
base = os.path.expanduser("~/lighthouse/home/junzhe/Projects/SG-VLN/dump2/")
poliformer_path = os.path.join(base, "benchmark_poliformer_jan28/result.json")
uninavid_path = os.path.join(base, "benchmark_uninavid_jan28/result.json")
uninavid_vln_path = os.path.join(base, "benchmark_uninavid_vln_jan29/result.json")
modular_agent_path = os.path.join(base, "benchmark_agent_jan29/result.json")
modular_agent_gtgoal_path = os.path.join(base, "benchmark_oracle_feb5/result.json")
episode_folder = os.path.join(os.path.dirname(os.path.dirname(base)), "robot_env/episodes")
innout_placenav_list = load_split(episode_folder, "innout_placenav")
gt_data = load_all_gt_data(episode_folder)

records = []
records += load_records(poliformer_path, method_name="poliformer")
records += load_records(uninavid_path, method_name="uninavid")
records += load_records(modular_agent_path, method_name="modular_agent")
records += load_records(uninavid_vln_path, method_name="uninavid_vln")
records += load_records(modular_agent_gtgoal_path, method_name="modular_agent_gtgoal")
print("Loaded records:", len(records))

# Infer numeric metric keys (you can override manually if you want)
metric_keys = numeric_keys(records)
summary = summarize_by_method(records, metric_keys)

# Pretty-print a small summary preview
for m, mets in summary.items():
    print("\n==", m, "==")
    for k in sorted(mets.keys())[:10]:
        s = mets[k]
        print(f"  {k}: mean={s['mean']:.4g}, std={s['std']:.4g}, n={s['n']}")


In [ ]:
# --------- 2.5) FIX + REWRITE result.json (success/spl/soft_spl) AND RELOAD ---------
# This block:
#   1) reads each original result.json
#   2) sets success = 1 if distance_to_goal < THRESH else 0
#   3) recomputes spl (fallback: spl=success if optimal path length is unavailable)
#   4) computes soft_spl = soft_success(distance)*path_length
#   5) writes a NEW json next to the original: result_fixed.json
#   6) reloads `records` from these fixed files (so the rest of this notebook uses the fixed metrics)

import json
import os

THRESH = 1.6

LSTAR_KEYS = [
    "shortest_path_length", "geodesic_distance", "shortest_path", "gt_path_length",
    "optimal_path_length", "shortest_len", "shortest_length",
]

def get_episode_gt(scene_id='grCommercial_scene6', episode_id=16):
    return gt_data[scene_id][episode_id]

def _try_float(x):
    try:
        return float(x)
    except Exception:
        return None

def _compute_success(distance_to_goal, thresh=THRESH):
    d = _try_float(distance_to_goal)
    if d is None:
        return 0.0
    return 1.0 if d < float(thresh) else 0.0

def _soft_success(distance_to_goal, thresh=THRESH):
    d = _try_float(distance_to_goal)
    if d is None:
        return 0.0
    return max(0.0, 1.0 - d / float(thresh))

def get_gt_path_length(r: dict=None, scene_id: str=None, episode_id: int=None, episode_label: str=None):
    if r is None:
        if episode_label is not None:
            scene_id = "_".join(episode_label.split("_")[:-1])
            episode_id = int(episode_label.split("_")[-1])
        gt_data = get_episode_gt(scene_id, episode_id)
    else:
        gt_data = get_episode_gt(r["scene_id"], r["episode_id"])
    goal_idx = gt_data["closest_goal_idx"]
    closest_goal = gt_data["goals"][goal_idx]
    return closest_goal["path_length"]

def _compute_spl(r: dict, thresh=THRESH):
    suc = _compute_success(r.get("distance_to_goal"), thresh=thresh)
    L = _try_float(r.get("path_length")) # agent path length
    Lstar = get_gt_path_length(r) # oracle path length
    eps = 1e-9
    if (L is not None) and (Lstar is not None):
        return float(suc) * (float(Lstar) / max(float(L), float(Lstar), eps))
    # Fallback when L* is not available in the record
    return float(suc)

def _compute_soft_spl(r: dict, thresh=THRESH):
    gt_path_length = get_gt_path_length(r)
    progress = 1-r.get("distance_to_goal")/gt_path_length
    progress = max(0, min(1, progress))
    spl = gt_path_length/max(gt_path_length, r.get("path_length"))
    soft_spl = progress*gt_path_length/max(gt_path_length, r.get("path_length"))
    # print(f"episode: {r.get('episode_label')}, progress: {progress}, path_length: {r.get('path_length')}, dtg: {r.get('distance_to_goal')}, gt_path_length: {gt_path_length}, soft_spl: {soft_spl}, spl: {spl}")
    return soft_spl

def rewrite_result_json(in_path: str, out_path: str, thresh=THRESH):
    obj = json.loads(open(in_path, "r", encoding="utf-8").read())
    recs = extract_records(obj)
    fixed = []
    for r in recs:
        rr = dict(r)
        rr["success"] = _compute_success(rr.get("distance_to_goal"), thresh=thresh)
        rr["spl"] = _compute_spl(rr, thresh=thresh)
        rr["soft_spl"] = _compute_soft_spl(rr, thresh=thresh)
        fixed.append(rr)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(fixed, f, indent=2)
    return out_path, len(fixed)

# --- rewrite 3 files (same as the 3 paths in block 2) ---
poliformer_fixed_path = os.path.join(os.path.dirname(poliformer_path), "result_fixed.json")
uninavid_fixed_path   = os.path.join(os.path.dirname(uninavid_path),   "result_fixed.json")
modular_agent_fixed_path = os.path.join(os.path.dirname(modular_agent_path), "result_fixed.json")
uninavid_vln_fixed_path = os.path.join(os.path.dirname(uninavid_vln_path), "result_fixed.json")
for _in, _out in [
    # (poliformer_path, poliformer_fixed_path),
    # (uninavid_path, uninavid_fixed_path),
    # (modular_agent_path, modular_agent_fixed_path),
    # (uninavid_vln_path, uninavid_vln_fixed_path),
]:
    if os.path.exists(_in):
        outp, n = rewrite_result_json(_in, _out, thresh=THRESH)
        print(f"Wrote {outp}  (episodes={n})")
    else:
        print(f"[WARN] Missing: {_in}")

# --- reload records from the FIXED jsons ---
records = []
if os.path.exists(poliformer_fixed_path):
    records += load_records(poliformer_fixed_path, method_name="poliformer")
if os.path.exists(uninavid_fixed_path):
    records += load_records(uninavid_fixed_path, method_name="uninavid")
if os.path.exists(modular_agent_fixed_path):
    records += load_records(modular_agent_fixed_path, method_name="modular_agent")
if os.path.exists(uninavid_vln_fixed_path):
    records += load_records(uninavid_vln_fixed_path, method_name="uninavid_vln")

print("Reloaded FIXED records:", len(records))

metric_keys = numeric_keys(records)
summary = summarize_by_method(records, metric_keys)
for m, mets in summary.items():
    print("\n==", m, "==")
    for k in ["success", "spl", "soft_spl", "distance_to_goal", "path_length"]:
        if k in mets:
            s = mets[k]
            print(f"  {k}: mean={s['mean']:.4g}, std={s['std']:.4g}, n={s['n']}")


In [ ]:
# calculate overlap episodes and overlap success_episodes

records = {
    "poliformer": load_records(poliformer_path.replace("result.json", "result_fixed.json"), method_name="poliformer"),
    "uninavid": load_records(uninavid_path.replace("result.json", "result_fixed.json"), method_name="uninavid"),
    "modular_agent": load_records(modular_agent_path.replace("result.json", "result_fixed.json"),   method_name="modular_agent")
}

episodes = [[r["episode_label"] for r in records[m]] for m in records]
overlap_episodes = list(set(episodes[0]) & set(episodes[1]) & set(episodes[2]))
success_episodes = [r["episode_label"] for r in records["poliformer"] if r["success"] == 1]
overlap_success_episodes = list(set(success_episodes) & set(overlap_episodes))


In [ ]:
# sample episodes
# results = []
# for episode_label in overlap_success_episodes:
#     scene_id = "_".join(episode_label.split("_")[:-1])
#     episode_id = int(episode_label.split("_")[-1])
#     results.append([episode_label, get_gt_path_length(scene_id=scene_id, episode_id=episode_id)])
# results.sort(key=lambda x: x[1], reverse=True)
# for r in results:
#     print(r)

records = []
if os.path.exists(poliformer_fixed_path):
    records += load_records(poliformer_fixed_path, method_name="poliformer")
if os.path.exists(uninavid_fixed_path):
    records += load_records(uninavid_fixed_path, method_name="uninavid")
if os.path.exists(modular_agent_fixed_path):
    records += load_records(modular_agent_fixed_path, method_name="modular_agent")

results = []
for episode_label in overlap_episodes:
    scene_id = "_".join(episode_label.split("_")[:-1])
    episode_id = int(episode_label.split("_")[-1])
    results.append([
        sum([r['success'] for r in records if r['episode_label']==episode_label]),
        get_gt_path_length(scene_id=scene_id, episode_id=episode_id),
        episode_label
    ])

scenes = {
    "indoor": lambda l: l.startswith("gr"),
    "outdoor_objnav": lambda l: l.startswith("vc") and not "store" in l,
    "outdoor_placenav": lambda l: l.startswith("vc") and "store" in l,
    "innout": lambda l: l.startswith("innout"),
}
difficulties = {
    "easy": lambda l: get_gt_path_length(episode_label=l) < 10,
    "medium": lambda l: 10 <= get_gt_path_length(episode_label=l) < 30,
    "hard": lambda l: get_gt_path_length(episode_label=l) >= 30,
}
results.sort(reverse=True)
sample_episodes = {}
for scene_type in scenes:
    for difficulty in difficulties:
        label = f"{scene_type}_{difficulty}"
        print(f"{label}:")
        count = 0
        for r in results:
            if scenes[scene_type](r[2]) and difficulties[difficulty](r[2]):
                print("  ",r)
                sample_episodes[r[2]] = label
                count += 1
            if count >= 3:
                break


In [ ]:
# # gather video path for the same episode
# import shutil
# methods_to_folders = {
#     "PoliFormer": poliformer_path.replace("result.json", ""),
#     "UniNaVid": uninavid_path.replace("result.json", ""),
#     "SGImagineNav": modular_agent_path.replace("result.json", ""),
# }

# download_folder = os.path.join(base, "download")
# os.makedirs(download_folder, exist_ok=True)
# print(f"Download folder: {download_folder}")
# for sample_episode in sample_episodes:
#     label = sample_episodes[sample_episode]
#     for method in methods_to_folders:
#         original_path = f"{methods_to_folders[method]}vid/{sample_episode}_obs_action_{method}.mp4"
#         if not os.path.exists(original_path):
#             original_path = f"{methods_to_folders[method]}vid/{sample_episode}_rgb_{method}.mp4"
#             if not os.path.exists(original_path):
#                 print(f"not found: {original_path}")
#                 continue
#         new_path = f"{download_folder}/{label}_{sample_episode}_{method}.mp4"
#         print(f"{method}: {original_path}")
#         shutil.copy(original_path, new_path)


In [ ]:
# zip the download folder
shutil.make_archive(os.path.join(base, "download"), "zip", download_folder)

In [ ]:
# categorize scenes and tasks
is_indoor = lambda r: r["scene_id"].startswith("gr")
is_outdoor = lambda r: r["scene_id"].startswith("vc")
is_outdoor_objnav = lambda r: is_outdoor(r) and not "store" in r["scene_id"]
is_outdoor_placenav = lambda r: is_outdoor(r) and "store" in r["scene_id"]
is_innout = lambda r: r["scene_id"].startswith("innout")
is_innout_placenav = lambda r: is_innout(r) and r["episode_label"] in innout_placenav_list
is_innout_objnav = lambda r: is_innout(r) and not r["episode_label"] in innout_placenav_list
is_objnav = lambda r: is_indoor(r) or is_outdoor_objnav(r) or is_innout_objnav(r)
is_placenav = lambda r: is_outdoor_placenav(r) or is_innout_placenav(r)
is_easy = lambda r: get_gt_path_length(r) < 10
is_medium = lambda r: 10 <= get_gt_path_length(r) < 30
is_hard = lambda r: get_gt_path_length(r) >= 30

In [ ]:
# (table 2) calculate metrics for each scene/task

records = {
    "poliformer": load_records(poliformer_path.replace("result.json", "result_fixed.json"), method_name="poliformer"),
    "uninavid": load_records(uninavid_path.replace("result.json", "result_fixed.json"), method_name="uninavid"),
    "modular_agent": load_records(modular_agent_path.replace("result.json", "result_fixed.json"),   method_name="modular_agent"),
    "uninavid_vln": load_records(uninavid_vln_path.replace("result.json", "result_fixed.json"),   method_name="uninavid_vln")
}

records_by_scene_task = {
    "alltask_allscene": {
        method: [r for r in records[method]]
        for method in records
    },
    "objnav_allscene": {
        method: [r for r in records[method] if is_objnav(r)]
        for method in records
    },
    "indoor_objnav": {
        method: [r for r in records[method] if is_objnav(r) and is_indoor(r)]
        for method in records
    },
    "outdoor_objnav": {
        method: [r for r in records[method] if is_objnav(r) and is_outdoor(r)]
        for method in records
    },
    "innout_objnav": {
        method: [r for r in records[method] if is_objnav(r) and is_innout(r)]
        for method in records
    },
    "placenav_allscene": {
        method: [r for r in records[method] if is_placenav(r)]
        for method in records
    },
    "outdoor_placenav": {
        method: [r for r in records[method] if is_placenav(r) and is_outdoor(r)]
        for method in records
    },
    "innout_placenav": {
        method: [r for r in records[method] if is_placenav(r) and is_innout(r)]
        for method in records
    }
}

metric_keys = ['success', 'spl']

for record_type in records_by_scene_task:
    tmp_records = []
    for method in records_by_scene_task[record_type]:
        tmp_records.extend(records_by_scene_task[record_type][method])
    # print summary
    summary = summarize_by_method(tmp_records, metric_keys)
    print(f"Summary for {record_type}:")
    for m, mets in summary.items():
        print(f"  {m}:")
        for k in mets:
            print(f"    {k}: {mets[k]['mean']*100:.4g}, {mets[k]['std']*100:.4g}, {mets[k]['n']}")


In [ ]:
# (table 4) calculate metrics for different difficulty levels

records = {
    "poliformer": load_records(poliformer_path.replace("result.json", "result_fixed.json"), method_name="poliformer"),
    "uninavid": load_records(uninavid_path.replace("result.json", "result_fixed.json"), method_name="uninavid"),
    "modular_agent": load_records(modular_agent_path.replace("result.json", "result_fixed.json"),   method_name="modular_agent"),
    "uninavid_vln": load_records(uninavid_vln_path.replace("result.json", "result_fixed.json"),   method_name="uninavid_vln")
}


records_by_scene_task = {
    "alltask_easy": {
        method: [r for r in records[method] if is_easy(r)]
        for method in records
    },
    "alltask_medium": {
        method: [r for r in records[method] if is_medium(r)]
        for method in records
    },
    "alltask_hard": {
        method: [r for r in records[method] if is_hard(r)]
        for method in records
    },
    "objnav_easy": {
        method: [r for r in records[method] if is_objnav(r) and is_easy(r)]
        for method in records
    },
    "objnav_medium": {
        method: [r for r in records[method] if is_objnav(r) and is_medium(r)]
        for method in records
    },
    "objnav_hard": {
        method: [r for r in records[method] if is_objnav(r) and is_hard(r)]
        for method in records
    },
    "placenav_easy": {
        method: [r for r in records[method] if is_placenav(r) and is_easy(r)]
        for method in records
    },
    "placenav_medium": {
        method: [r for r in records[method] if is_placenav(r) and is_medium(r)]
        for method in records
    },
    "placenav_hard": {
        method: [r for r in records[method] if is_placenav(r) and is_hard(r)]
        for method in records
    },
}

metric_keys = ['success', 'spl']

for record_type in records_by_scene_task:
    tmp_records = []
    for method in records_by_scene_task[record_type]:
        tmp_records.extend(records_by_scene_task[record_type][method])
    # print summary
    summary = summarize_by_method(tmp_records, metric_keys)
    print(f"Summary for {record_type}:")
    for m, mets in summary.items():
        print(f"  {m}:")
        for k in mets:
            print(f"    {k}: {mets[k]['mean']*100:.4g}, {mets[k]['std']*100:.4g}, {mets[k]['n']}")


In [ ]:
# (table 5) calculate metrics for different difficulty levels

records = {
    "uninavid": load_records(uninavid_path.replace("result.json", "result_fixed.json"), method_name="uninavid"),
    "uninavid_vln": load_records(uninavid_vln_path.replace("result.json", "result_fixed.json"),   method_name="uninavid_vln")
}


records_by_scene_task = {
    "objnav_allscene": {
        method: [r for r in records[method] if is_objnav(r)]
        for method in records
    },
    "objnav_indoor": {
        method: [r for r in records[method] if is_objnav(r) and is_indoor(r)]
        for method in records
    },
    "objnav_outdoor": {
        method: [r for r in records[method] if is_objnav(r) and is_outdoor(r)]
        for method in records
    },
    "objnav_innout": {
        method: [r for r in records[method] if is_objnav(r) and is_innout(r)]
        for method in records
    },
    "placenav_allscene": {
        method: [r for r in records[method] if is_placenav(r)]
        for method in records
    },
    "placenav_indoor": {
        method: [r for r in records[method] if is_placenav(r) and is_indoor(r)]
        for method in records
    },
    "placenav_outdoor": {
        method: [r for r in records[method] if is_placenav(r) and is_outdoor(r)]
        for method in records
    },
    "placenav_innout": {
        method: [r for r in records[method] if is_placenav(r) and is_innout(r)]
        for method in records
    },
    "alltask_allscene": {
        method: [r for r in records[method]]
        for method in records
    },
    "alltask_indoor": {
        method: [r for r in records[method] if is_indoor(r)]
        for method in records
    },
    "alltask_outdoor": {
        method: [r for r in records[method] if is_outdoor(r)]
        for method in records
    },
    "alltask_innout": {
        method: [r for r in records[method] if is_innout(r)]
        for method in records
    },
}

metric_keys = ['success', 'spl']

for record_type in records_by_scene_task:
    tmp_records = []
    for method in records_by_scene_task[record_type]:
        tmp_records.extend(records_by_scene_task[record_type][method])
    # print summary
    summary = summarize_by_method(tmp_records, metric_keys)
    print(f"Summary for {record_type}:")
    for m, mets in summary.items():
        print(f"  {m}:")
        for k in mets:
            print(f"    {k}: {mets[k]['mean']*100:.4g}, {mets[k]['std']*100:.4g}, {mets[k]['n']}")


In [25]:
file_list = """
indoor_easy_grCommercial_scene29_13_PoliFormer.mp4
indoor_easy_grCommercial_scene29_13_SGImagineNav.mp4
indoor_easy_grCommercial_scene29_13_UniNaVid.mp4
indoor_medium_grCommercial_scene13_12_PoliFormer.mp4
indoor_medium_grCommercial_scene13_12_SGImagineNav.mp4
indoor_medium_grCommercial_scene13_12_UniNaVid.mp4
indoor_medium_grCommercial_scene26_6_PoliFormer.mp4
indoor_medium_grCommercial_scene26_6_SGImagineNav.mp4
indoor_medium_grCommercial_scene26_6_UniNaVid.mp4
innout_easy_innout_barcelona_15_PoliFormer.mp4
innout_easy_innout_barcelona_15_SGImagineNav.mp4
innout_easy_innout_barcelona_15_UniNaVid.mp4
innout_hard_innout_ny_2_PoliFormer.mp4
innout_hard_innout_ny_2_SGImagineNav.mp4
innout_hard_innout_ny_2_UniNaVid.mp4
innout_medium_innout_amsterdam_1_PoliFormer.mp4
innout_medium_innout_amsterdam_1_SGImagineNav.mp4
innout_medium_innout_amsterdam_1_UniNaVid.mp4
innout_medium_innout_berlin_8_PoliFormer.mp4
innout_medium_innout_berlin_8_SGImagineNav.mp4
innout_medium_innout_berlin_8_UniNaVid.mp4
outdoor_objnav_easy_vc_austin_5_PoliFormer.mp4
outdoor_objnav_easy_vc_austin_5_SGImagineNav.mp4
outdoor_objnav_easy_vc_austin_5_UniNaVid.mp4
outdoor_objnav_hard_vc_barcelona_9_PoliFormer.mp4
outdoor_objnav_hard_vc_barcelona_9_SGImagineNav.mp4
outdoor_objnav_hard_vc_barcelona_9_UniNaVid.mp4
outdoor_objnav_medium_vc_amsterdam_17_PoliFormer.mp4
outdoor_objnav_medium_vc_amsterdam_17_SGImagineNav.mp4
outdoor_objnav_medium_vc_amsterdam_17_UniNaVid.mp4
outdoor_placenav_easy_vc_amsterdam_store_14_PoliFormer.mp4
outdoor_placenav_easy_vc_amsterdam_store_14_SGImagineNav.mp4
outdoor_placenav_easy_vc_amsterdam_store_14_UniNaVid.mp4
outdoor_placenav_easy_vc_baltimore_store_16_PoliFormer.mp4
outdoor_placenav_easy_vc_baltimore_store_16_SGImagineNav.mp4
outdoor_placenav_easy_vc_baltimore_store_16_UniNaVid.mp4
outdoor_placenav_hard_vc_amsterdam_store_5_PoliFormer.mp4
outdoor_placenav_hard_vc_amsterdam_store_5_SGImagineNav.mp4
outdoor_placenav_hard_vc_amsterdam_store_5_UniNaVid.mp4
outdoor_placenav_hard_vc_baltimore_store_8_PoliFormer.mp4
outdoor_placenav_hard_vc_baltimore_store_8_SGImagineNav.mp4
outdoor_placenav_hard_vc_baltimore_store_8_UniNaVid.mp4
outdoor_placenav_hard_vc_berlin_store_18_PoliFormer.mp4
outdoor_placenav_hard_vc_berlin_store_18_SGImagineNav.mp4
outdoor_placenav_hard_vc_berlin_store_18_UniNaVid.mp4
"""
episode_list = file_list.strip().split("\n")
episode_list = [f.replace("placenav_", "").replace("objnav_", "") for f in episode_list]
episode_list = ["_".join(f.split(".")[0].split("_")[2:5]) for f in episode_list]
episode_list = list(set(episode_list))
episode_list = sorted(episode_list, reverse=True)
for episode in episode_list:
    print(episode)


vc_berlin_store
vc_barcelona_9
vc_baltimore_store
vc_austin_5
vc_amsterdam_store
vc_amsterdam_17
innout_ny_2
innout_berlin_8
innout_barcelona_15
innout_amsterdam_1
grCommercial_scene29_13
grCommercial_scene26_6
grCommercial_scene13_12
